<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
Modell-Kosten
</b></font> </br></p>

---

In [ ]:
#@title 🛠️ Umgebung einrichten { display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

# LangSmith Env-Vars vor LangChain-Imports setzen
# Hinweis: Tracing ist hier bewusst aktiv (anders als in anderen Modulen),
# damit Abschnitt 4 und 5 echte Runs zum Auswerten haben.

# ── Stdlib ───────────────────────────────────────────────────────────────────
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M15-Modell-Kosten"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

# ── Projekt-Utilities ───────────────────────────────────────────────
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    mermaid,
)
from genai_lib.model_config import BASELINE, WORKER

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---

Jeder LLM-Aufruf verbraucht Tokens und verursacht dadurch Kosten – pro Anfrage meist gering, über ein Projekt oder einen Kurs hinweg aber durchaus spürbar. Kostenkontrolle gehört ebenso zum bewussten Umgang mit Modellen wie die Wahl der passenden Rolle (`BASELINE`, `WORKER`, `JUDGE`, ...).

Es gibt zwei Wege, Kosten zu ermitteln:

| Weg | Ansatz | Wann sinnvoll |
|---|---|---|
| **Python & Token** | Tokens direkt aus der Antwort lesen (`usage_metadata`), mit einer Preistabelle verrechnen | einzelne Aufrufe, schnelle Schätzung, keine Abhängigkeit von LangSmith |
| **LangSmith** | Kosten über den `Client` aus getrackten Runs abfragen | projektweite Auswertung über Zeit, Vergleich mehrerer Modelle/Rollen |

Beide Wege ergänzen sich: Der Python-Weg funktioniert immer, auch ohne Tracing. Der LangSmith-Weg liefert eine zentrale Übersicht, sobald mit Tracing gearbeitet wird – genau das ist in diesem Notebook aktiviert (`LANGSMITH_TRACING="true"`), damit die folgenden Abschnitte mit echten Runs arbeiten können.

In [2]:
#@markdown   <p><font size="4" color='green'>  Zwei Wege zur Kostenermittlung</font> </br></p>

diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    A[LLM-Aufruf] --> B[usage_metadata]
    B --> C[Preistabelle]
    C --> D[Kosten je Aufruf]
    A --> E[LangSmith Tracing]
    E --> F[Client.list_runs]
    F --> G[Kosten je Projekt]
    D --> H[Kostenuebersicht]
    G --> H
"""
mermaid(diagram, width=1100)

# 2 | Tokens zählen in Python
---

Jede `AIMessage`-Antwort von `init_chat_model()` enthält ein `usage_metadata`-Feld mit den verbrauchten Tokens:

| Feld | Bedeutung |
|---|---|
| `input_tokens` | Tokens im Prompt (System- und User-Message) |
| `output_tokens` | Tokens in der generierten Antwort |
| `total_tokens` | Summe aus Input und Output |

Bei Reasoning-Modellen wie `gpt-5.4` stecken zusätzliche interne Denkschritte in `output_tokens` – Details dazu liefert `output_token_details`, falls das Modell sie zurückgibt.

Bei wiederholt genutzten, langen Prompts (z. B. ein großer System-Prompt oder RAG-Kontext) markieren manche Anbieter einen Teil der Input-Tokens als **gecacht** und berechnen dafür einen reduzierten Preis. Diese Tokens stecken bereits in `input_tokens`, werden aber zusätzlich unter `input_token_details.cache_read` ausgewiesen – nur wer dieses Feld separat verrechnet, bekommt die tatsächlichen Kosten.

Für eine reine Vorab-Schätzung (vor dem eigentlichen API-Aufruf) eignet sich `tiktoken`. Für die tatsächlich abgerechneten Tokens ist `usage_metadata` nach dem Aufruf die zuverlässigere Quelle.

In [3]:
from langchain.chat_models import init_chat_model

prompt = "Erkläre in zwei Sätzen, was Retrieval Augmented Generation ist."

llm_baseline = init_chat_model(BASELINE)
llm_worker = init_chat_model(WORKER)

run_cfg_base = {
    "run_name": "M15_Kosten_Baseline",
    "tags": ["m15", "kosten", "baseline"],
    "metadata": {"notebook": "M15", "rolle": "BASELINE", "modell": BASELINE},
}
run_cfg_worker = {
    "run_name": "M15_Kosten_Worker",
    "tags": ["m15", "kosten", "worker"],
    "metadata": {"notebook": "M15", "rolle": "WORKER", "modell": WORKER},
}

antwort_baseline = llm_baseline.invoke(prompt, config=run_cfg_base)
antwort_worker = llm_worker.invoke(prompt, config=run_cfg_worker)

zeilen = []
for name, antwort in [("BASELINE", antwort_baseline), ("WORKER", antwort_worker)]:
    zeilen.append(f"**{name}**: `{antwort.usage_metadata}`")

mprint("\n\n".join(zeilen))

**BASELINE**: `{'input_tokens': 21, 'output_tokens': 62, 'total_tokens': 83, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}`

**WORKER**: `{'input_tokens': 21, 'output_tokens': 68, 'total_tokens': 89, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}`

# 3 | Kosten aus Tokens berechnen
---

⚠️ **Warnung:** Modellpreise ändern sich regelmäßig. Die folgende Tabelle enthält **Platzhalterwerte** (USD pro 1 Million Tokens). Vor einer echten Kostenauswertung immer die aktuelle Preisliste unter [platform.openai.com/pricing](https://platform.openai.com/docs/pricing) prüfen und die Werte hier ersetzen.

In [ ]:
import pandas as pd

# Platzhalterwerte in USD pro 1 Mio. Tokens - vor echter Nutzung aktualisieren.
# "cached_input": Preis für Input-Tokens, die der Anbieter aus dem Prompt-Cache bedient (meist stark rabattiert).
PREISE_PRO_1M_TOKEN = {
    "gpt-5.4-nano": {"input": 0.05, "cached_input": 0.005, "output": 0.40},
    "gpt-5.4-mini": {"input": 0.25, "cached_input": 0.025, "output": 2.00},
    "gpt-5.4":      {"input": 1.25, "cached_input": 0.125, "output": 10.00},
}


def berechne_kosten(usage_metadata: dict, modell_kurzname: str) -> float:
    """Berechnet die Kosten eines Aufrufs aus usage_metadata und Preistabelle (USD).

    Gecachte Input-Tokens (`input_token_details.cache_read`) werden zum reduzierten
    `cached_input`-Preis abgerechnet, da sie in `input_tokens` bereits mitgezählt sind.
    """
    if not usage_metadata:
        raise ValueError("usage_metadata fehlt (None oder leer) - liegt eine echte AIMessage-Antwort vor?")

    if modell_kurzname not in PREISE_PRO_1M_TOKEN:
        bekannte_modelle = ", ".join(sorted(PREISE_PRO_1M_TOKEN))
        raise ValueError(
            f"Unbekanntes Modell: {modell_kurzname}. "
            f"Ergänze PREISE_PRO_1M_TOKEN oder nutze eines davon: {bekannte_modelle}"
        )

    fehlende_felder = [
        feld for feld in ["input_tokens", "output_tokens"]
        if feld not in usage_metadata
    ]
    if fehlende_felder:
        raise ValueError(f"usage_metadata enthält nicht alle Token-Felder: {fehlende_felder}")

    preise = PREISE_PRO_1M_TOKEN[modell_kurzname]
    cache_tokens = usage_metadata.get("input_token_details", {}).get("cache_read", 0)
    normale_input_tokens = usage_metadata["input_tokens"] - cache_tokens

    input_kosten = normale_input_tokens / 1_000_000 * preise["input"]
    cache_kosten = cache_tokens / 1_000_000 * preise["cached_input"]
    output_kosten = usage_metadata["output_tokens"] / 1_000_000 * preise["output"]
    return round(input_kosten + cache_kosten + output_kosten, 6)


kosten_baseline = berechne_kosten(antwort_baseline.usage_metadata, "gpt-5.4-nano")
kosten_worker = berechne_kosten(antwort_worker.usage_metadata, "gpt-5.4-mini")

df_vergleich = pd.DataFrame([
    {"rolle": "BASELINE", "modell": "gpt-5.4-nano", "total_tokens": antwort_baseline.usage_metadata["total_tokens"], "kosten_usd": kosten_baseline},
    {"rolle": "WORKER", "modell": "gpt-5.4-mini", "total_tokens": antwort_worker.usage_metadata["total_tokens"], "kosten_usd": kosten_worker},
])
df_vergleich

In [ ]:
import plotly.express as px

fig = px.bar(
    df_vergleich,
    x="rolle",
    y="kosten_usd",
    text_auto=True,
    labels={"rolle": "Rolle", "kosten_usd": "Kosten (USD)"},
    title="Kosten je Modellrolle",
)
fig.show()

# 4 | Kosten über LangSmith abfragen (optional)
---

🔹 **Optional:** Dieser Abschnitt baut auf LangSmith auf – einem separaten Tracing-Dienst, der über die Grundlagen aus Abschnitt 2/3 hinausgeht. Für das Grundverständnis von Modell-Kosten (Tokens lesen, mit Preistabelle verrechnen) ist er nicht nötig. Wer nur mit `usage_metadata` arbeiten möchte, kann direkt zu Abschnitt A (Aufgabe) springen.

Die Aufrufe aus Abschnitt 2 wurden bereits unter dem LangSmith-Projekt `M15-Modell-Kosten` getrackt (siehe Setup-Zelle). Über den `Client` lassen sich alle Runs eines Projekts inklusive Kostenfeldern abfragen.

⚠️ **Hinweis:** `total_cost`, `prompt_cost` und `completion_cost` sind nur befüllt, wenn im LangSmith-Workspace ein Pricing-Mapping für das jeweilige Modell hinterlegt ist. Ohne Mapping liefert LangSmith `None` – die Tokens (`total_tokens` etc.) sind davon unabhängig immer vorhanden.

In [ ]:
from datetime import datetime, timedelta, timezone
from typing import Any

from langsmith import Client

client = Client()


def list_projects() -> list[Any]:
    """Alle LangSmith-Projekte auflisten (Name + ID)."""
    return list(client.list_projects())


def get_project_costs(project_name: str, days: int = 1) -> list[dict[str, Any]]:
    """Kosten aller Runs eines Projekts über die letzten `days` Tage."""
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=days)

    rows = []
    runs = client.list_runs(project_name=project_name, start_time=start, end_time=end)

    for run in runs:
        rows.append({
            "run_id": str(run.id),
            "name": run.name,
            "run_type": run.run_type,
            "start_time": run.start_time,
            "total_tokens": getattr(run, "total_tokens", None),
            "prompt_tokens": getattr(run, "prompt_tokens", None),
            "completion_tokens": getattr(run, "completion_tokens", None),
            "total_cost": getattr(run, "total_cost", None),
            "prompt_cost": getattr(run, "prompt_cost", None),
            "completion_cost": getattr(run, "completion_cost", None),
            "project_name": project_name,
        })

    return rows


projekte = [p.name for p in list_projects()]
mprint(f"Vorhandene Projekte: {projekte}")

runs_m15 = get_project_costs("M15-Modell-Kosten", days=1)
pd.DataFrame(runs_m15)

# 5 | Kosten-Export & Auswertung (optional)
---

Für eine dauerhafte Auswertung lassen sich die Kostendaten als CSV exportieren und mit `pandas` aggregieren – z. B. Kosten je Modell oder je Rolle über einen längeren Zeitraum.

In [7]:
def export_project_costs_to_csv(
    project_name: str,
    days: int = 30,
    output_file: str = "langsmith_costs.csv",
) -> pd.DataFrame:
    """Kosten eines Projekts exportieren und als DataFrame zurückgeben."""
    rows = get_project_costs(project_name, days=days)

    if not rows:
        print("Keine Runs gefunden.")
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df.to_csv(output_file, index=False, encoding="utf-8")

    print(f"Kosten exportiert nach: {output_file}")
    print(f"Anzahl Runs: {len(df)}")
    if "total_cost" in df.columns and df["total_cost"].notna().any():
        print(f"Summe total_cost: {df['total_cost'].sum():.6f}")

    return df


df_m15 = export_project_costs_to_csv("M15-Modell-Kosten", days=1, output_file="langsmith_costs_m15.csv")

if not df_m15.empty:
    df_m15.groupby("name")[["total_tokens"]].sum()

Kosten exportiert nach: langsmith_costs_m15.csv
Anzahl Runs: 2
Summe total_cost: 0.000403


**🎯 FAZIT:** Kosten lassen sich auf zwei Wegen ermitteln – direkt aus `usage_metadata` mit einer eigenen Preistabelle (funktioniert immer, auch ohne Tracing) oder zentral über LangSmith-Runs (praktisch für projektweite Auswertungen über Zeit). Für schnelle Schätzungen einzelner Aufrufe reicht der Python-Weg; für einen Überblick über viele Aufrufe hinweg lohnt sich LangSmith.

---

# A | Aufgabe
---

**Grundlagen**

Frage über `list_projects()` alle vorhandenen LangSmith-Projekte ab und gib ihre Namen aus. Wähle eines der Kursprojekte und rufe `get_project_costs()` dafür für die letzten 7 Tage auf.

**Aufbau**

Erweitere `berechne_kosten()` um ein drittes Modell aus `PREISE_PRO_1M_TOKEN` (z. B. `gpt-5.4`) und vergleiche die Kosten für denselben Prompt über alle drei Rollen hinweg in einer Tabelle.

**Vertiefung**

Baue eine Auswertung über mehrere Aufrufe hinweg: Sammle für 5–10 verschiedene Prompts jeweils Tokens und Kosten je Rolle und ermittle, ab welcher Prompt-Länge sich der Wechsel von `BASELINE` auf `WORKER` in Tokens/Kosten bemerkbar macht. Gib das Ergebnis als kleine Tabelle oder Diagramm aus.

# B | Dokumente zum Weiterlesen
---

- [OpenAI Pricing](https://platform.openai.com/docs/pricing)
- [LangSmith: Usage and Cost Graphs](https://docs.smith.langchain.com/observability/how_to_guides/dashboards)
- [LangChain: Track token usage](https://python.langchain.com/docs/how_to/chat_token_usage_tracking/)